## Preprocessing and data cleaning

- [**dataset1**](https://doi.org/10.6084/m9.figshare.25771164): PSS-10, GAD-7, PHQ-9 totals and survey bands 
- [**dataset2**](https://data.mendeley.com/datasets/5ny7cth7vw/1): DASS-21 (doubled subscale scores) and parsed demographics

In [3]:
import pandas as pd
from pathlib import Path

from utils import (
    print_dataset_summary,
    extract_number,
    age_int_to_bracket,
    cgpa_float_to_band,
    semester_int_to_academic_year,
    dass21_stress_score_to_level,
    dass21_anxiety_score_to_level,
    dass21_depression_score_to_level,
)

### load raw tables

In [4]:
# assume notebook is run from proj_root/mhp/
ROOT = Path.cwd().parent
RAW_DATA = ROOT / "raw_data"
DATA = ROOT / "mhp" / "data"

dataset1 = pd.read_csv(
    RAW_DATA / "MHP Dataset of University Students_processed.csv"
)
dataset2 = pd.read_csv(
    RAW_DATA / "Dataset for Analyzing University Students Behaviour.csv",
    sep=";",
    encoding="utf-8",
)

for key, (title, frame) in {
    "dataset1": ("MHP Dataset of University Students", dataset1),
    "dataset2": (
        "Dataset for Analyzing University Students Behaviour",
        dataset2,
    ),
}.items():
    print_dataset_summary(key, title, frame)


dataset1 — MHP Dataset of University Students
Shape: 2,028 rows, 39 columns

Column names:
	1. Age
	2. Gender
	3. University
	4. Department
	5. Academic_Year
	6. Current_CGPA
	7. waiver_or_scholarship
	8. PSS1
	9. PSS2
	10. PSS3
	11. PSS4
	12. PSS5
	13. PSS6
	14. PSS7
	15. PSS8
	16. PSS9
	17. PSS10
	18. Stress Value
	19. Stress Label
	20. GAD1
	21. GAD2
	22. GAD3
	23. GAD4
	24. GAD5
	25. GAD6
	26. GAD7
	27. Anxiety Value
	28. Anxiety Label
	29. PHQ1
	30. PHQ2
	31. PHQ3
	32. PHQ4
	33. PHQ5
	34. PHQ6
	35. PHQ7
	36. PHQ8
	37. PHQ9
	38. Depression Value
	39. Depression Label

Missing values: none

Per-column type-value summary:
Age
	dtype=str | enum | 5 distinct labels
	missing: 0 (0.00%)
Gender
	dtype=str | enum | 3 distinct labels
	missing: 0 (0.00%)
University
	dtype=str | enum | 15 distinct labels
	missing: 0 (0.00%)
Department
	dtype=str | enum | 12 distinct labels
	missing: 0 (0.00%)
Academic_Year
	dtype=str | enum | 5 distinct labels
	missing: 0 (0.00%)
Current_CGPA
	dtype=str | en

### constants shared by both datasets

In [5]:
AGE_ORDER = ["<18", "18-22", "23-26", "27-30", ">30"]
GENDER_LABELS = ["female", "male", "other"]

# department codes for dataset1
# dataset2 uses only "not_recorded" but categories must match
DEPARTMENT_CODES = sorted(
    {
        "bio_sciences",
        "business",
        "eng_cs",
        "eng_civil",
        "eng_eee",
        "eng_mech",
        "eng_other",
        "env_life_sciences",
        "law",
        "liberal_arts",
        "other",
        "pharmacy",
        "not_recorded",
    }
)

ACADEMIC_YEAR_ORDER = ["unknown", "first", "second", "third", "fourth", "other"]
CGPA_ORDER = [
    "other",
    "<2.50",
    "2.50-2.99",
    "3.00-3.39",
    "3.40-3.79",
    "3.80-4.00",
]

SEVERITY_ORDER = [
    "none",
    "minimal",
    "mild",
    "moderate",
    "moderately_severe",
    "severe",
]

GENDER_FROM_RAW = {
    "Female": "female",
    "Male": "male",
    "Prefer not to say": "other",
}

OUTPUT_COLUMNS = [
    "age",
    "gender",
    "department",
    "academic_year",
    "cgpa",
    "stress_level",
    "stress_z_score",
    "anxiety_level",
    "anxiety_z_score",
    "depression_level",
    "depression_z_score",
]

### dataset1: column renames, value maps, output schema

In [6]:
DATASET1_COLUMN_MAP = {
    "Age": "age",
    "Gender": "gender",
    "Department": "department",
    "Academic_Year": "academic_year",
    "Current_CGPA": "cgpa",
    "Stress Value": "pss10_stress_score",
    "Stress Label": "stress_level",
    "Anxiety Value": "gad7_anxiety_score",
    "Anxiety Label": "anxiety_level",
    "Depression Value": "phq9_depression_score",
    "Depression Label": "depression_level",
}

DATASET1_VALUE_MAPS = {
    "age": {
        "Below 18": "<18",
        "18-22": "18-22",
        "23-26": "23-26",
        "27-30": "27-30",
        "Above 30": ">30",
    },
    "department": {
        "Biological Sciences": "bio_sciences",
        "Business and Entrepreneurship Studies": "business",
        "Engineering - CS / CSE / CSC / Similar to CS": "eng_cs",
        "Engineering - Civil Engineering / Similar to CE": "eng_civil",
        "Engineering - EEE/ ECE / Similar to EEE": "eng_eee",
        "Engineering - Mechanical Engineering / Similar to ME": "eng_mech",
        "Engineering - Other": "eng_other",
        "Environmental and Life Sciences": "env_life_sciences",
        "Law and Human Rights": "law",
        "Liberal Arts and Social Sciences": "liberal_arts",
        "Other": "other",
        "Pharmacy and Public Health": "pharmacy",
    },
    "academic_year": {
        "First Year or Equivalent": "first",
        "Second Year or Equivalent": "second",
        "Third Year or Equivalent": "third",
        "Fourth Year or Equivalent": "fourth",
        "Other": "other",
    },
    "cgpa": {
        "Other": "other",
        "Below 2.50": "<2.50",
        "2.50 - 2.99": "2.50-2.99",
        "3.00 - 3.39": "3.00-3.39",
        "3.40 - 3.79": "3.40-3.79",
        "3.80 - 4.00": "3.80-4.00",
    },
    "stress_level": {
        "Low Stress": "minimal",
        "Moderate Stress": "moderate",
        "High Perceived Stress": "severe",
    },
    "anxiety_level": {
        "Minimal Anxiety": "minimal",
        "Mild Anxiety": "mild",
        "Moderate Anxiety": "moderate",
        "Severe Anxiety": "severe",
    },
    "depression_level": {
        "No Depression": "none",
        "Minimal Depression": "minimal",
        "Mild Depression": "mild",
        "Moderate Depression": "moderate",
        "Moderately Severe Depression": "moderately_severe",
        "Severe Depression": "severe",
    },
}

DATASET1_OUTPUT_COLUMNS = [
    "age",
    "gender",
    "department",
    "academic_year",
    "cgpa",
    "pss10_stress_score",
    "stress_level",
    "stress_z_score",
    "gad7_anxiety_score",
    "anxiety_level",
    "anxiety_z_score",
    "phq9_depression_score",
    "depression_level",
    "depression_z_score",
]

### dataset1: build `dataset1_clean`

In [7]:
dataset1_clean = dataset1[list(DATASET1_COLUMN_MAP.keys())].copy()
dataset1_clean = dataset1_clean.rename(columns=DATASET1_COLUMN_MAP)

dataset1_clean["gender"] = dataset1_clean["gender"].map(GENDER_FROM_RAW)

for col, mapping in DATASET1_VALUE_MAPS.items():
    dataset1_clean[col] = dataset1_clean[col].map(mapping)

for col, cats in (
    ("age", AGE_ORDER),
    ("academic_year", ACADEMIC_YEAR_ORDER),
    ("cgpa", CGPA_ORDER),
    ("stress_level", SEVERITY_ORDER),
    ("anxiety_level", SEVERITY_ORDER),
    ("depression_level", SEVERITY_ORDER),
):
    dataset1_clean[col] = pd.Categorical(
        dataset1_clean[col], categories=cats, ordered=True
    )

dataset1_clean["gender"] = pd.Categorical(
    dataset1_clean["gender"], GENDER_LABELS, ordered=False
)
dataset1_clean["department"] = pd.Categorical(
    dataset1_clean["department"], DEPARTMENT_CODES, ordered=False
)

# calculate z-scores
dataset1_clean["stress_z_score"] = (
    dataset1_clean["pss10_stress_score"]
    - dataset1_clean["pss10_stress_score"].mean()
) / dataset1_clean["pss10_stress_score"].std()
dataset1_clean["anxiety_z_score"] = (
    dataset1_clean["gad7_anxiety_score"]
    - dataset1_clean["gad7_anxiety_score"].mean()
) / dataset1_clean["gad7_anxiety_score"].std()
dataset1_clean["depression_z_score"] = (
    dataset1_clean["phq9_depression_score"]
    - dataset1_clean["phq9_depression_score"].mean()
) / dataset1_clean["phq9_depression_score"].std()

# reorder columns
dataset1_clean = dataset1_clean[DATASET1_OUTPUT_COLUMNS]

dataset1_clean.info()
dataset1_clean.head()

<class 'pandas.DataFrame'>
RangeIndex: 2028 entries, 0 to 2027
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   age                    2028 non-null   category
 1   gender                 2028 non-null   category
 2   department             2028 non-null   category
 3   academic_year          2028 non-null   category
 4   cgpa                   2028 non-null   category
 5   pss10_stress_score     2028 non-null   int64   
 6   stress_level           2028 non-null   category
 7   stress_z_score         2028 non-null   float64 
 8   gad7_anxiety_score     2028 non-null   int64   
 9   anxiety_level          2028 non-null   category
 10  anxiety_z_score        2028 non-null   float64 
 11  phq9_depression_score  2028 non-null   int64   
 12  depression_level       2028 non-null   category
 13  depression_z_score     2028 non-null   float64 
dtypes: category(8), float64(3), int64(3)
memory usage: 

,age,gender,department,academic_year,cgpa,pss10_stress_score,stress_level,stress_z_score,gad7_anxiety_score,anxiety_level,anxiety_z_score,phq9_depression_score,depression_level,depression_z_score
0,18-22,female,eng_cs,second,2.50-2.99,29,severe,0.888514,15,severe,0.483266,20,severe,0.835727
1,18-22,male,eng_cs,third,3.00-3.39,24,moderate,0.148329,12,moderate,-0.062832,19,moderately_severe,0.685753
2,18-22,male,eng_cs,third,3.00-3.39,15,moderate,-1.184004,0,minimal,-2.247223,0,none,-2.163750
3,18-22,male,eng_cs,third,3.00-3.39,17,moderate,-0.887930,10,moderate,-0.426897,14,moderate,-0.064116
4,18-22,male,eng_cs,second,2.50-2.99,32,severe,1.332624,14,moderate,0.301233,20,severe,0.835727


### DASS-21 (Depression, Anxiety, and Stress Scale)

The [DASS-21](https://www.scu.edu.au/media/scu-dep/current-students/services/counselling/documents/DASS.pdf) measures Depression, Anxiety, and Stress through 21 items (7 questions per category). Answers are scored 0 to 3.

**The rating scale is as follows:**
- **0:** Did not apply to me at all — *NEVER*
- **1:** Applied to me to some degree, or some of the time — *SOMETIMES*
- **2:** Applied to me to a considerable degree, or a good part of time — *OFTEN*
- **3:** Applied to me very much, or most of the time — *ALMOST ALWAYS*

#### Depression (7 items): Measures lack of interest, hopelessness, and dysphoria.
- I couldn’t seem to experience any positive feelings at all
- I found it difficult to work up the initiative to do things
- I felt that I had nothing to look forward to
- I felt down-hearted and blue
- I was unable to become enthusiastic about anything
- I felt I wasn’t worth much as a person
- I felt that life was meaningless

#### Anxiety (7 items): Measures physical arousal and situational panic.
- I was aware of the dryness of my mouth
- I experienced breathing difficulty (e.g. excessively rapid breathing, breathlessness in the absence of physical exertion)
- I experienced trembling (e.g. in the hands)
- I was worried about situations in which I might panic and make a fool of myself
- I felt I was close to panic
- I was aware of the action of my heart in the absence of physical exertion (e.g. sense of heart rate increase, heart missing a beat)
- I felt scared without any good reason

#### Stress (7 items): Measures tension, agitation, and difficulty relaxing.
- I found it hard to wind down
- I tended to over-react to situations
- I felt that I was using a lot of nervous energy
- I found myself getting agitated
- I found it difficult to relax
- I was intolerant of anything that kept me from getting on with what I was doing
- I felt that I was rather touchy

| Severity Level     | Depression Score | Anxiety Score | Stress Score |
| :---------------- | :--------------: | :-----------: | :----------: |
| Normal            | 0 - 9            | 0 - 7         | 0 - 14       |
| Mild              | 10 - 13          | 8 - 9         | 15 - 18      |
| Moderate          | 14 - 20          | 10 - 14       | 19 - 25      |
| Severe            | 21 - 27          | 15 - 19       | 26 - 33      |
| Extremely Severe  | 28+              | 20+           | 34+          |

### dataset2: column renames, constants, value maps, output schema

In [8]:
# source .csv column names
# DASS items = last 21 columns
DATASET2_COLUMN_MAP = {
    "Semester( number like: 1st, 2nd...)": "semester_raw",
    "Age( Years: 18, 19....)": "age_raw",
    "Gender": "gender",
    "Academic performance(CGPA like: 3.5.....)": "cgpa_raw",
}

# DASS-21 answers to 0–3 (must match .csv wording)
DATASET2_DASS_ANSWER_MAP = {
    "Did not apply to me at all": 0,
    "Applied to me to some degree, or some of the time": 1,
    "Applied to me to a considerable degree or a good part of time": 2,
    "Applied to me very much or most of the time": 3,
}

# standard DASS-21 ordering
DATASET2_DASS_STRESS_INDICES = [1, 6, 8, 11, 12, 14, 18]
DATASET2_DASS_ANXIETY_INDICES = [2, 4, 7, 9, 15, 19, 20]
DATASET2_DASS_DEPRESSION_INDICES = [3, 5, 10, 13, 16, 17, 21]

DATASET2_OUTPUT_COLUMNS = [
    "age_parsed",
    "age",
    "gender",
    "department",
    "semester_parsed",
    "academic_year",
    "cgpa_parsed",
    "cgpa",
    "dass21_stress_score",
    "stress_level",
    "stress_z_score",
    "dass21_anxiety_score",
    "anxiety_level",
    "anxiety_z_score",
    "dass21_depression_score",
    "depression_level",
    "depression_z_score",
]

### dataset2: build `dataset2_intermediate` and  `dataset2_clean`

In [9]:
dass21_src_cols = list(dataset2.columns[-21:])
DASS_COLS_MAP = {
    raw: f"dass_{i:02d}" for i, raw in enumerate(dass21_src_cols, start=1)
}
dataset2_wide = dataset2[
    list(DATASET2_COLUMN_MAP.keys()) + dass21_src_cols
].copy()
dataset2_wide = dataset2_wide.rename(
    columns={**DATASET2_COLUMN_MAP, **DASS_COLS_MAP}
)

print("DASS mapping:")
for k, v in DASS_COLS_MAP.items():
    print(f"- {v}: {k.strip()}")

DASS mapping:
- dass_01: I found it hard to wind down
- dass_02: I was aware of the dryness of my mouth
- dass_03: I couldn’t seem to experience any positive feelings at all
- dass_04: I experienced breathing difficulty (e.g. excessively rapid breathing, breathlessness in the absence of physical exertion)
- dass_05: I found it difficult to work up the initiative to do things
- dass_06: I tended to over-react to situations
- dass_07: I experienced trembling (e.g. in the hands)
- dass_08: I felt that I was using a lot of nervous energy
- dass_09: I was worried about situations in which I might panic and make a fool of myself
- dass_10: I felt that I had nothing to look forward to
- dass_11: I found myself getting agitated
- dass_12: I found it difficult to relax
- dass_13: I felt down-hearted and blue
- dass_14: I was intolerant of anything that kept me from getting on with what I was doing
- dass_15: I felt I was close to panic
- dass_16: I was unable to become enthusiastic about anythi

#### parse age

In [10]:
dataset2_wide["age_parsed"] = (
    dataset2_wide["age_raw"]
    .apply(lambda x: round(extract_number(x)))
    .astype("Int64")
)

print("mapping: raw age -> parsed age")
mapping_df = (
    dataset2_wide[["age_raw", "age_parsed"]]
    .drop_duplicates()
    .sort_values("age_parsed")
)
display(mapping_df)

mapping: raw age -> parsed age


,age_raw,age_parsed
81,18,18
27,19,19
335,19 years 11 months 24 days,19
22,20,20
334,20 years 9 months 30 days,20
1,21,21
2,22,22
6,"22,5",22
4,23,23
0,24,24


#### parse semester

In [11]:
# manually defined mapping after inspection of semester values
_WORD_TO_SEMESTER = {
    "second": 2,
    "last": 9,  # map to "other"
}


def parse_semester(value: str) -> int | None:
    n = extract_number(value)
    if n is not None:
        return int(n)
    clean = str(value).strip().lower()
    for word, num in _WORD_TO_SEMESTER.items():
        if word in clean:
            return num
    return None


dataset2_wide["semester_parsed"] = (
    dataset2_wide["semester_raw"].apply(parse_semester).astype("Int64")
)

print("mapping: raw semester -> parsed semester")
mapping_df = (
    dataset2_wide[["semester_raw", "semester_parsed"]]
    .drop_duplicates()
    .sort_values("semester_parsed")
)
display(mapping_df)

mapping: raw semester -> parsed semester


,semester_raw,semester_parsed
87,1 st,1
65,1st,1
54,2nd,2
199,2nd,2
202,2,2
119,second,2
106,2nd semester,2
317,3rd,3
5,4th,4
251,Complete 4th semester,4


In [12]:
# manual fixes (invalid or year-like semester strings)
BAD_SEM_IDX = [129, 200, 307, 340]
dataset2_wide.loc[BAD_SEM_IDX, "semester_parsed"] = pd.NA

#### parse cgpa

In [13]:
dataset2_wide["cgpa_parsed"] = (
    dataset2_wide["cgpa_raw"].apply(extract_number).astype("Float64")
)

print("mapping: raw cgpa -> parsed cgpa")
mapping_df = (
    dataset2_wide[["cgpa_raw", "cgpa_parsed"]]
    .drop_duplicates()
    .sort_values("cgpa_parsed")
)
display(mapping_df)

mapping: raw cgpa -> parsed cgpa


,cgpa_raw,cgpa_parsed
155,00,0.0
205,"1,27",1.27
147,"1,52",1.52
63,"2,09",2.09
104,"2,26",2.26
...,...,...
70,NaN,<NA>
144,"I am a fresher, first semester student.",<NA>
146,not get yet,<NA>
148,no cgpa,<NA>


#### parse DASS scores

In [14]:
dass_cols = [c for c in dataset2_wide.columns if c.startswith("dass_")]
for c in dass_cols:
    dataset2_wide[c] = (
        dataset2_wide[c].map(DATASET2_DASS_ANSWER_MAP).astype("Int64")
    )

stress_cols = [f"dass_{i:02d}" for i in DATASET2_DASS_STRESS_INDICES]
anxiety_cols = [f"dass_{i:02d}" for i in DATASET2_DASS_ANXIETY_INDICES]
depression_cols = [f"dass_{i:02d}" for i in DATASET2_DASS_DEPRESSION_INDICES]

dataset2_wide["dass21_stress_score"] = (
    dataset2_wide[stress_cols].sum(axis=1, min_count=7) * 2
)
dataset2_wide["dass21_anxiety_score"] = (
    dataset2_wide[anxiety_cols].sum(axis=1, min_count=7) * 2
)
dataset2_wide["dass21_depression_score"] = (
    dataset2_wide[depression_cols].sum(axis=1, min_count=7) * 2
)

# calculate z-scores
dataset2_wide["stress_z_score"] = (
    dataset2_wide["dass21_stress_score"]
    - dataset2_wide["dass21_stress_score"].mean()
) / dataset2_wide["dass21_stress_score"].std()
dataset2_wide["anxiety_z_score"] = (
    dataset2_wide["dass21_anxiety_score"]
    - dataset2_wide["dass21_anxiety_score"].mean()
) / dataset2_wide["dass21_anxiety_score"].std()
dataset2_wide["depression_z_score"] = (
    dataset2_wide["dass21_depression_score"]
    - dataset2_wide["dass21_depression_score"].mean()
) / dataset2_wide["dass21_depression_score"].std()

#### map parsed values to brackets to match dataset1

In [15]:
dataset2_wide["department"] = "not_recorded"
dataset2_wide["gender"] = (
    dataset2_wide["gender"].map(GENDER_FROM_RAW).fillna("other")
)

dataset2_wide["age"] = dataset2_wide["age_parsed"].map(age_int_to_bracket)
dataset2_wide["cgpa"] = dataset2_wide["cgpa_parsed"].map(cgpa_float_to_band)
dataset2_wide["academic_year"] = dataset2_wide["semester_parsed"].map(
    semester_int_to_academic_year
)

dataset2_wide["stress_level"] = dataset2_wide["dass21_stress_score"].map(
    dass21_stress_score_to_level
)
dataset2_wide["anxiety_level"] = dataset2_wide["dass21_anxiety_score"].map(
    dass21_anxiety_score_to_level
)
dataset2_wide["depression_level"] = dataset2_wide[
    "dass21_depression_score"
].map(dass21_depression_score_to_level)

for col, cats in (
    ("age", AGE_ORDER),
    ("academic_year", ACADEMIC_YEAR_ORDER),
    ("cgpa", CGPA_ORDER),
    ("stress_level", SEVERITY_ORDER),
    ("anxiety_level", SEVERITY_ORDER),
    ("depression_level", SEVERITY_ORDER),
):
    dataset2_wide[col] = pd.Categorical(
        dataset2_wide[col], categories=cats, ordered=True
    )

dataset2_wide["gender"] = pd.Categorical(
    dataset2_wide["gender"], categories=GENDER_LABELS, ordered=False
)
dataset2_wide["department"] = pd.Categorical(
    dataset2_wide["department"], categories=DEPARTMENT_CODES, ordered=False
)

dataset2_wide.info()

dataset2_clean = dataset2_wide[DATASET2_OUTPUT_COLUMNS].copy()
dataset2_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 351 entries, 0 to 350
Data columns (total 41 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   semester_raw             351 non-null    str     
 1   age_raw                  351 non-null    str     
 2   gender                   351 non-null    category
 3   cgpa_raw                 346 non-null    str     
 4   dass_01                  351 non-null    Int64   
 5   dass_02                  351 non-null    Int64   
 6   dass_03                  351 non-null    Int64   
 7   dass_04                  343 non-null    Int64   
 8   dass_05                  351 non-null    Int64   
 9   dass_06                  351 non-null    Int64   
 10  dass_07                  351 non-null    Int64   
 11  dass_08                  351 non-null    Int64   
 12  dass_09                  351 non-null    Int64   
 13  dass_10                  351 non-null    Int64   
 14  dass_11              

### merge datasets and save outputs

In [16]:
dataset1_clean.to_csv(DATA / "dataset1_clean.csv", index=False)
dataset2_wide.to_csv(DATA / "dataset2_intermediate.csv", index=False)
dataset2_clean.to_csv(DATA / "dataset2_clean.csv", index=False)

dataset1_merge = dataset1_clean[OUTPUT_COLUMNS].copy()
dataset2_merge = dataset2_clean[OUTPUT_COLUMNS].copy()

dataset1_merge["source_dataset"] = "dataset1"
dataset2_merge["source_dataset"] = "dataset2"

# ensure "source_dataset" is categorical so that its consistent with the other cols
SOURCE_DATASET_CATEGORIES = ["dataset1", "dataset2"]
dataset1_merge["source_dataset"] = pd.Categorical(
    dataset1_merge["source_dataset"], categories=SOURCE_DATASET_CATEGORIES
)
dataset2_merge["source_dataset"] = pd.Categorical(
    dataset2_merge["source_dataset"], categories=SOURCE_DATASET_CATEGORIES
)

mhp_dataset = pd.concat([dataset1_merge, dataset2_merge], ignore_index=True)
mhp_dataset.info()
mhp_dataset.head()

mhp_dataset.to_csv(DATA / "mhp_dataset.csv", index=False)
# save to .parquet to maintain the cols dtype
mhp_dataset.to_parquet(DATA / "mhp_dataset.parquet")

<class 'pandas.DataFrame'>
RangeIndex: 2379 entries, 0 to 2378
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   age                 2379 non-null   category
 1   gender              2379 non-null   category
 2   department          2379 non-null   category
 3   academic_year       2379 non-null   category
 4   cgpa                2379 non-null   category
 5   stress_level        2379 non-null   category
 6   stress_z_score      2379 non-null   Float64 
 7   anxiety_level       2371 non-null   category
 8   anxiety_z_score     2371 non-null   Float64 
 9   depression_level    2379 non-null   category
 10  depression_z_score  2379 non-null   Float64 
 11  source_dataset      2379 non-null   category
dtypes: Float64(3), category(9)
memory usage: 86.3 KB
